# Register a functional plane to its matched anatomy slice and warp its ROI masks

**Pipeline for one fish/plane:**

1. Load the plane's suite2p mean image + ROI masks (`stat.npy`), and the
   fish's raw anatomy stack.
2. Find the anatomy depth and scale this plane best matches, using
   [`registration/plane_matching.py`](../plane_matching.py)
3. Register the plane onto that anatomy slice: NCC placement, then `ants`
   Rigid and Affine inside a square around that placement (the plane plus
   10%). It returns the fitted transform and the aligned plane image.
4. Warp the plane's suite2p ROI masks through the fitted transform, so each
   ROI's aligned pixel coordinates live in anatomy space.
5. Write the 8-bit anatomy NRRD (canonical anatomy space) used for anatomy →
   reference brain registration.

Reference implementations: merge of Danin and Lukas pipeline. The registration and ROI warping follow Danin Dharmaperwira's `ddharmap-ANTs` pipeline (`spatial.py`,
`_ants_rigid_affine_in_plane_result`; `matching.py`,
`_resample_labels_ants_nn`).

## 1. Imports & environment setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from skimage.segmentation import find_boundaries

# Walk upward from the current working directory to the repo root (marked
# by utils.py, always present at the top level) and add it to sys.path, so
# `registration` is importable below no matter where Jupyter was launched from.
repo_root = Path.cwd()
while not (repo_root / "utils.py").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from preprocessing.spatial_preprocessing import ACQUISITION_XY_FRAME, anatomy_z_spacing_um, preprocess_anatomy, resolve_polarity
from registration.image_utils import norm01, read_pixel_size_um
from registration.plane_matching import match_plane_to_anatomy
from registration.plane_registration import (
    register_plane_to_anatomy,
    warp_roi_masks_to_anatomy,
    write_registration_metadata,
    load_suite2p_plane,
    read_functional_xy_frame,
)
from utils import init_experiment_tree

## 2. Fish setup

In [ ]:
fish_id = "L331_f01"

#base_data_path = Path("/Volumes/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/07_Data/Matilde/Microscopy")
base_data_path = Path("/Volumes/LAB-MATI/Lausanne/2p/Microscopy")
fish_folder = base_data_path / fish_id

# Creates any missing standard folders (e.g. 02_reg/07_2pf-a/logs); existing ones are left as they are.
# Also returns the tree's key paths (raw anatomy, suite2p output, ...), used below.
experiment_paths = init_experiment_tree(base_data_path, fish_id)

## 3. Load anatomy stack
Load the raw anatomy stack for this fish and retrieve its metadata (channels, pixel size, Z step, etc.)

In [ ]:
anatomy_dir = experiment_paths["raw_2p_anatomy"]

# Excludes ex-vivo scans; keeps the one in-vivo anatomy stack from the same session.
anatomy_candidates = [
    path
    for path in sorted(anatomy_dir.glob("*.tif*"))
    if "ex_vivo" not in path.stem.lower() and "exvivo" not in path.stem.lower()
]
if len(anatomy_candidates) != 1:
    raise ValueError(f"Expected exactly one raw in-vivo anatomy TIFF under {anatomy_dir}; found {anatomy_candidates}")
anatomy_path = anatomy_candidates[0]

# Load anatomy stack
# Read every page into a 3D array, plus ScanImage's own acquisition metadata.
with tifffile.TiffFile(anatomy_path) as anatomy_tiff:
    anatomy_pages = np.stack([page.asarray() for page in anatomy_tiff.pages], axis=0)
    scanimage_metadata = (anatomy_tiff.scanimage_metadata or {}).get("FrameData", {})

# ScanImage's own record of which channel(s) were saved.
channel_save = scanimage_metadata.get("SI.hChannels.channelSave")
anatomy_channel_count = len(channel_save) if isinstance(channel_save, list) else 1

if anatomy_channel_count > 1:
    print(f"WARNING: anatomy TIFF has {anatomy_channel_count} channels -- only the first channel is being used.")
    # Channels are interleaved fastest-varying, so every Nth page is channel 0.
    anatomy_stack = anatomy_pages[0::anatomy_channel_count].astype(np.float32)
else:
    anatomy_stack = anatomy_pages.astype(np.float32)

# Read the pixel size in microns from the TIFF metadata
anatomy_pixel_size_um = read_pixel_size_um(anatomy_path)

# Distance between slices, from the raw metadata CSV (`step_size_um_anatomy`; ScanImage doesn't record it).
anatomy_z_step_um, anatomy_z_step_sources = anatomy_z_spacing_um(experiment_paths["raw_2p_metadata"])

**Orientation check:** the raw anatomy is in the microscope's acquisition
orientation, so the suite2p data has to be too. This reads the fish's
preprocessing record (canonical manifest or `functional_preprocessing` metadata) and
stops if the functional movies were flipped into the canonical orientation;
registering those against the unflipped anatomy would put them in the wrong
place. Fish with no record (preprocessed before either existed) were never
flipped.

In [ ]:
functional_xy_frame, functional_xy_frame_source = read_functional_xy_frame(fish_folder)
print(f"Functional XY frame: {functional_xy_frame} ({functional_xy_frame_source})")
if functional_xy_frame != ACQUISITION_XY_FRAME:
    raise ValueError(
        f"{fish_id}'s suite2p data was preprocessed in the flipped (canonical) XY orientation, "
        f"per {functional_xy_frame_source}, but this notebook registers it against the unflipped raw anatomy. "
        "Registering the two together would place the planes wrongly; pre-flipped fish aren't supported yet."
    )

## 4. Process all planes

Finds every plane folder under this fish's suite2p output, then for each
plane: loads its suite2p outputs, matches it to the anatomy stack,
registers it, and warps its ROI masks — collecting everything into
`plane_results`.

### 4.1 Discover this fish's planes

In [ ]:
plane_folders = experiment_paths["analysis_suite2p"].glob("plane*")
plane_indices = sorted(int(folder.name.replace("plane", "")) for folder in plane_folders)
print(f"Found {len(plane_indices)} planes for {fish_id}: {plane_indices}")

### 4.2 Find best matches for each functional plane in the anatomy stack

Sharpens both images, slides the reference over every anatomy slice at a range of
scales, and scores each position with normalized cross-correlation
(`TM_CCORR_NORMED`, ~0-1, higher = more shared structure). The scale search
follows Danin's production settings (0.50-1.50, then three finer passes).
Returns a dict; we only use:

- `best_z` *(int)* — winning slice index, used below to pick `anatomy_slice`.
- `score` *(float)* — correlation at that depth; diagnostic only, eyeball the plot if it's low.
- `best_z_subslice` *(float)* — sub-slice-precision version of the same peak; diagnostic only.
- `scale` *(float)* — resize factor that puts the plane on the anatomy's pixel grid; `register_plane_to_anatomy` (docstring) explains how it's used.

In [ ]:
# Input provenance shared by every plane, recorded in each plane's metadata.
anatomy_sources = {
    "anatomy_path": str(anatomy_path),
    "anatomy_shape_zyx": list(anatomy_stack.shape),
    "anatomy_pixel_size_um": anatomy_pixel_size_um,
    "anatomy_z_step_um": anatomy_z_step_um,
    "functional_xy_frame": functional_xy_frame,
    "functional_xy_frame_source": functional_xy_frame_source,
}

# plane_-prefixed names throughout, so nothing here gets confused with a
# specific plane if you rerun a single one later in section 6.
plane_results = []
for plane_idx in plane_indices:
    plane_prefix = f"{fish_id}_plane{plane_idx}"
    plane_suite2p_folder = experiment_paths["analysis_suite2p"] / f"plane{plane_idx}"

    # Load the suite2p mean image and ROI masks for this plane.
    plane_reference_image, plane_stat_array = load_suite2p_plane(plane_suite2p_folder, plane_prefix)

    # Match this plane to the anatomy stack, returning the best-matching depth and NCC score.
    plane_match = match_plane_to_anatomy(plane_reference_image, anatomy_stack)
    plane_best_depth = plane_match["best_z"]
    plane_anatomy_slice = anatomy_stack[plane_best_depth]  # from the original, unsharpened stack

    # Transform files go to the stage's transMatrices/ folder, under a per-plane prefix.
    plane_transform_prefix = fish_folder / f"02_reg/07_2pf-a/transMatrices/{plane_prefix}_to_anatomy_"
    # Register the plane to its matched anatomy slice, returning the ANTs registration result (warped image, transform, etc.).
    plane_registration = register_plane_to_anatomy(
        plane_reference_image, plane_anatomy_slice, anatomy_pixel_size_um, plane_match["scale"], plane_transform_prefix
    )
    # Warp the plane's ROI masks to anatomy space using the ANTs transform.
    plane_warped_rois = warp_roi_masks_to_anatomy(
        plane_stat_array, plane_registration, plane_reference_image.shape, plane_best_depth
    )

    # Write a JSON file with the plane's registration metadata, including the ANTs transform and warped ROIs.
    write_registration_metadata(
        plane_registration, plane_match, plane_stat_array, plane_warped_rois,
        {**anatomy_sources, "suite2p_folder": str(plane_suite2p_folder)}, fish_id,
        fish_folder / f"02_reg/07_2pf-a/logs/{plane_prefix}_to_anatomy_registration_metadata.json",
    )

    plane_results.append({
        "plane_index": plane_idx,
        "file_prefix": plane_prefix,
        "reference_image": plane_reference_image,
        "match": plane_match,
        "best_depth": plane_best_depth,
        "anatomy_slice": plane_anatomy_slice,
        "registration": plane_registration,
        "warped_image": plane_registration["warped_image"],
        "fixed_image": plane_registration["fixed_image"],
        "warped_rois": plane_warped_rois,
    })

## 5. QC plots for all planes

### 5.1 Registration overlays, one panel per plane

In [ ]:
grid_ncols = 3
grid_nrows = -(-len(plane_results) // grid_ncols)  # ceil division
# "compressed" layout packs the square images tightly, without white gaps.
registration_overlays_figure, registration_overlays_axes = plt.subplots(
    grid_nrows, grid_ncols, figsize=(6 * grid_ncols, 6 * grid_nrows), squeeze=False, layout="compressed"
)
for panel_index, result in enumerate(plane_results):
    row, col = divmod(panel_index, grid_ncols)
    axis = registration_overlays_axes[row][col]
    axis.imshow(result["fixed_image"], cmap="Reds", vmax=0.5)
    axis.imshow(result["warped_image"], cmap="Greens", alpha=0.6)
    # Zoom to the square ANTs registered in (fixed_image is 0 outside it); -0.5 aligns
    # with pixel edges, and the y axis runs top-down.
    square_x0, square_y0, square_x1, square_y1 = result["registration"]["fixed_region_xyxy"]
    axis.set_xlim(square_x0 - 0.5, square_x1 - 0.5)
    axis.set_ylim(square_y1 - 0.5, square_y0 - 0.5)
    axis.set_title(f"plane {result['plane_index']}, z={result['best_depth']}, post_ncc={result['registration']['post_ncc']:.2f}", fontsize=13)
    axis.axis("off")
# Hide any unused trailing panels (e.g. 4 planes leaves 2 empty in a 3x2 grid).
for panel_index in range(len(plane_results), grid_nrows * grid_ncols):
    row, col = divmod(panel_index, grid_ncols)
    registration_overlays_axes[row][col].axis("off")
registration_overlays_figure.suptitle(f"{fish_id} — registration overlays", fontsize=15, fontweight="bold")

#registration_overlays_path = fish_folder / "04_plots" / f"{fish_id}_all_planes_registration_overlays.png"
#Path(registration_overlays_path).parent.mkdir(parents=True, exist_ok=True)
#registration_overlays_figure.savefig(registration_overlays_path, dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Depth-match score profiles, one panel per plane

In [ ]:
PROFILE_LINE_COLOR = "#275d8c"  # blue NCC depth profile
BEST_DEPTH_COLOR = "#c1121f"  # red marker for the chosen depth

grid_ncols = 3
grid_nrows = -(-len(plane_results) // grid_ncols)  # ceil division
depth_profiles_figure, depth_profiles_axes = plt.subplots(
    grid_nrows, grid_ncols, figsize=(12, 2.9 * grid_nrows), sharey=True, squeeze=False, constrained_layout=True
)
for panel_index, result in enumerate(plane_results):
    row, col = divmod(panel_index, grid_ncols)
    axis = depth_profiles_axes[row][col]
    depth_scores = result["match"]["scores"]
    best_depth = result["best_depth"]
    axis.plot(depth_scores, color=PROFILE_LINE_COLOR, linewidth=1.5)
    axis.axvline(best_depth, color=BEST_DEPTH_COLOR, linestyle="--", linewidth=1.2)
    axis.scatter([best_depth], [depth_scores[best_depth]], color=BEST_DEPTH_COLOR, s=24, zorder=3)
    axis.set_title(f"plane {result['plane_index']}: best z={best_depth}", fontsize=10)
    axis.set_xlabel("Anatomy depth (slice index)")
    if col == 0:
        axis.set_ylabel("NCC score")
    axis.spines[["top", "right"]].set_visible(False)
# Hide any unused trailing panels (e.g. 5 planes leave 1 empty in a 3x2 grid).
for panel_index in range(len(plane_results), grid_nrows * grid_ncols):
    row, col = divmod(panel_index, grid_ncols)
    depth_profiles_axes[row][col].axis("off")
depth_profiles_figure.suptitle(f"{fish_id} — depth match score profiles")

#depth_profiles_path = fish_folder / "04_plots" / f"{fish_id}_all_planes_depth_score_profiles.png"
#Path(depth_profiles_path).parent.mkdir(parents=True, exist_ok=True)
#depth_profiles_figure.savefig(depth_profiles_path, dpi=150, bbox_inches="tight")
plt.show()

### 5.3 ROI overlays, one panel per plane

In [ ]:
ROI_ZOOM_MARGIN_PX = 20  # zoom to the ROIs' bounding box plus this margin, in anatomy pixels
FUNCTIONAL_OUTLINE_COLOR = "#51eaea"  # ROI outlines

grid_ncols = 3
grid_nrows = -(-len(plane_results) // grid_ncols)  # ceil division
# "compressed" layout packs the images tightly, without white gaps.
roi_overlays_figure, roi_overlays_axes = plt.subplots(
    grid_nrows, grid_ncols, figsize=(6 * grid_ncols, 6 * grid_nrows), squeeze=False, layout="compressed"
)
for panel_index, result in enumerate(plane_results):
    row, col = divmod(panel_index, grid_ncols)
    axis = roi_overlays_axes[row][col]
    # All warped ROIs in one label image (ROI position + 1, 0 = background).
    roi_labels = np.zeros(result["anatomy_slice"].shape, dtype=np.int32)
    for roi_number, roi in enumerate(result["warped_rois"], start=1):
        roi_labels[roi["ypix_aligned"], roi["xpix_aligned"]] = roi_number
    # Gray anatomy as RGB, with every ROI's outline painted cyan.
    roi_overlay = np.repeat(norm01(result["anatomy_slice"], 1, 99)[..., None], 3, axis=2)
    roi_overlay[find_boundaries(roi_labels, mode="inner")] = to_rgb(FUNCTIONAL_OUTLINE_COLOR)
    axis.imshow(roi_overlay)
    # Zoom to the ROIs' bounding box plus a margin (y axis runs top-down).
    roi_rows, roi_cols = np.nonzero(roi_labels)
    if roi_rows.size > 0:
        axis.set_xlim(roi_cols.min() - ROI_ZOOM_MARGIN_PX, roi_cols.max() + ROI_ZOOM_MARGIN_PX)
        axis.set_ylim(roi_rows.max() + ROI_ZOOM_MARGIN_PX, roi_rows.min() - ROI_ZOOM_MARGIN_PX)
    axis.set_title(f"plane {result['plane_index']} — {len(result['warped_rois'])} ROIs", fontsize=13)
    axis.axis("off")
# Hide any unused trailing panels (e.g. 4 planes leaves 2 empty in a 3x2 grid).
for panel_index in range(len(plane_results), grid_nrows * grid_ncols):
    row, col = divmod(panel_index, grid_ncols)
    roi_overlays_axes[row][col].axis("off")
roi_overlays_figure.suptitle(f"{fish_id} — ROI overlays", fontsize=15, fontweight="bold")

#roi_overlays_path = fish_folder / "04_plots" / f"{fish_id}_all_planes_roi_overlays.png"
#Path(roi_overlays_path).parent.mkdir(parents=True, exist_ok=True)
#roi_overlays_figure.savefig(roi_overlays_path, dpi=150, bbox_inches="tight")
plt.show()

### 5.4 Registration scores per plane

- `ncc_placement_score`: peak score of the sliding search that picks where the plane goes on the anatomy (translation only, before ANTs). High means the starting position was found confidently.
- `pre_ants_ncc` / `post_ncc`: the same correlation (mean-subtracted, only where the plane has data) before and after ANTs. `post_ncc` above `pre_ants_ncc` means ANTs improved the fit.
- `valid_fraction`: share of the anatomy slice covered by the plane.
- `peak_delta`: best depth score minus the second-highest score.
- `peak_zscore`: how far the best depth score stands above the mean over all depths, in standard deviations.

The second-highest score is often the best slice's neighbour, so a small
`peak_delta` doesn't necessarily mean two separate peaks. Check 5.2.

In [ ]:
registration_score_rows = []
for result in plane_results:
    depth_scores = np.asarray(result["match"]["scores"], dtype=np.float64)
    best_score = depth_scores[result["best_depth"]]
    # np.partition puts the two largest values last; [-2] is the runner-up.
    second_best_score = np.partition(depth_scores, -2)[-2]
    registration_score_rows.append({
        "plane": result["plane_index"],
        "best_depth": result["best_depth"],
        "ncc_placement_score": result["registration"]["ncc_placement"]["score"],
        "pre_ants_ncc": result["registration"]["pre_ants_ncc"],
        "post_ncc": result["registration"]["post_ncc"],
        "valid_fraction": result["registration"]["valid_fraction"],
        "peak_delta": best_score - second_best_score,
        # + 1e-6 avoids a divide-by-zero on a flat profile.
        "peak_zscore": (best_score - depth_scores.mean()) / (depth_scores.std() + 1e-6),
    })
registration_score_table = pd.DataFrame(registration_score_rows).set_index("plane")
registration_score_table.round(3)

## 6. Inspect one plane

Everything below reads from `plane_results` (section 4) for one chosen plane — nothing is recomputed.

In [ ]:
inspect_plane_index = plane_indices[0]  # edit this to inspect a different plane
inspect_result = next(result for result in plane_results if result["plane_index"] == inspect_plane_index)

### 6.1 Suite2p mean image, matched anatomy, and depth-match score profile

In [ ]:
INSPECT_CLIP_PERCENTILES = (2, 98)  # display contrast: (low, high) percentile clip
PROFILE_LINE_COLOR = "#275d8c"  # blue NCC depth profile (Danin's QC colours)
BEST_DEPTH_COLOR = "#c1121f"  # red marker for the chosen depth

inspect_depth_scores = inspect_result["match"]["scores"]
inspect_best_depth = inspect_result["best_depth"]
# Crop the anatomy to where the registered plane has data (warped_image is 0 elsewhere).
footprint_rows, footprint_cols = np.nonzero(inspect_result["warped_image"])
if footprint_rows.size > 0:
    inspect_anatomy_crop = inspect_result["anatomy_slice"][footprint_rows.min():footprint_rows.max() + 1, footprint_cols.min():footprint_cols.max() + 1]
else:
    inspect_anatomy_crop = inspect_result["anatomy_slice"]

inspect_figure, inspect_axes = plt.subplot_mosaic(
    [["mean_image", "anatomy"], ["profile", "profile"]], figsize=(12, 9), height_ratios=[3, 1], constrained_layout=True
)
# Percentile-clip for display only -- a few bright pixels otherwise wash out the images.
inspect_axes["mean_image"].imshow(norm01(inspect_result["reference_image"], *INSPECT_CLIP_PERCENTILES), cmap="gray")
inspect_axes["mean_image"].set_title(f"{inspect_result['file_prefix']} — suite2p mean image")
inspect_axes["anatomy"].imshow(norm01(inspect_anatomy_crop, *INSPECT_CLIP_PERCENTILES), cmap="gray")
inspect_axes["anatomy"].set_title(f"Matched anatomy slice — z={inspect_best_depth} (plane footprint)")
inspect_axes["mean_image"].axis("off")
inspect_axes["anatomy"].axis("off")

profile_axis = inspect_axes["profile"]
profile_axis.plot(inspect_depth_scores, color=PROFILE_LINE_COLOR, linewidth=1.5)
profile_axis.axvline(inspect_best_depth, color=BEST_DEPTH_COLOR, linestyle="--", linewidth=1.2, label=f"best z={inspect_best_depth}")
profile_axis.scatter([inspect_best_depth], [inspect_depth_scores[inspect_best_depth]], color=BEST_DEPTH_COLOR, s=24, zorder=3)
profile_axis.set_xlabel("Anatomy depth (slice index)")
profile_axis.set_ylabel("NCC score")
profile_axis.set_title("Depth match score profile")
profile_axis.spines[["top", "right"]].set_visible(False)
profile_axis.legend(frameon=False)
plt.show()

### 6.2 Registration overlay

In [ ]:
overlay_figure, overlay_axis = plt.subplots(figsize=(8, 8))
overlay_axis.imshow(inspect_result["fixed_image"], cmap="Reds", vmax=0.5)
overlay_axis.imshow(inspect_result["warped_image"], cmap="Greens", alpha=0.6)
# Zoom to the square ANTs registered in (fixed_image is 0 outside it); -0.5 aligns
# with pixel edges, and the y axis runs top-down.
inspect_square_x0, inspect_square_y0, inspect_square_x1, inspect_square_y1 = inspect_result["registration"]["fixed_region_xyxy"]
overlay_axis.set_xlim(inspect_square_x0 - 0.5, inspect_square_x1 - 0.5)
overlay_axis.set_ylim(inspect_square_y1 - 0.5, inspect_square_y0 - 0.5)
overlay_axis.set_title(f"{inspect_result['file_prefix']}, anatomy z={inspect_result['best_depth']}, post_ncc={inspect_result['registration']['post_ncc']:.2f}")
overlay_axis.axis("off")
plt.show()

### 6.3 ROI overlay

In [ ]:
ROI_ZOOM_MARGIN_PX = 20  # zoom to the ROIs' bounding box plus this margin, in anatomy pixels
FUNCTIONAL_OUTLINE_COLOR = (0.05, 0.95, 1.0)  # cyan ROI outlines (Danin's QC colour)

# All warped ROIs in one label image (ROI position + 1, 0 = background).
inspect_roi_labels = np.zeros(inspect_result["anatomy_slice"].shape, dtype=np.int32)
for roi_number, roi in enumerate(inspect_result["warped_rois"], start=1):
    inspect_roi_labels[roi["ypix_aligned"], roi["xpix_aligned"]] = roi_number
# Gray anatomy as RGB, with every ROI's outline painted cyan.
inspect_roi_overlay = np.repeat(norm01(inspect_result["anatomy_slice"], 1, 99)[..., None], 3, axis=2)
inspect_roi_overlay[find_boundaries(inspect_roi_labels, mode="inner")] = to_rgb(FUNCTIONAL_OUTLINE_COLOR)

roi_overlay_figure, roi_overlay_axis = plt.subplots(figsize=(8, 8))
roi_overlay_axis.imshow(inspect_roi_overlay)
# Zoom to the ROIs' bounding box plus a margin (y axis runs top-down).
inspect_roi_rows, inspect_roi_cols = np.nonzero(inspect_roi_labels)
if inspect_roi_rows.size > 0:
    roi_overlay_axis.set_xlim(inspect_roi_cols.min() - ROI_ZOOM_MARGIN_PX, inspect_roi_cols.max() + ROI_ZOOM_MARGIN_PX)
    roi_overlay_axis.set_ylim(inspect_roi_rows.max() + ROI_ZOOM_MARGIN_PX, inspect_roi_rows.min() - ROI_ZOOM_MARGIN_PX)
roi_overlay_axis.set_title(f"{inspect_result['file_prefix']} — {len(inspect_result['warped_rois'])} ROIs warped to anatomy z={inspect_result['best_depth']}")
roi_overlay_axis.axis("off")
plt.show()

## 7. Write the 8-bit anatomy NRRD for anatomy → reference registration

Builds the anatomy in canonical anatomy space with
`preprocessing.spatial_preprocessing.preprocess_anatomy` (Danin's convention:
polarity X/Y flip, Z reversal, 750x750 grid, uint8), at the same path and name
his canonical preprocessing uses. Polarity comes from the fish's raw metadata
(`fish_orientation`); if that's missing, set `FISH_POLARITY` below. An existing
NRRD is never overwritten: delete it by hand to rebuild.

ROIs aren't converted here: they're mapped later, when the anatomy → reference
transform is applied, starting from each plane's registration metadata in
`02_reg/07_2pf-a/logs/`.


In [ ]:
FISH_POLARITY = 'south'  # "north" or "south"; only used if the raw metadata has no fish_orientation

# preprocess_anatomy reads every TIFF page as a slice, so a multi-channel stack would mix channels.
if anatomy_channel_count > 1:
    raise ValueError(f"{anatomy_path.name} has {anatomy_channel_count} channels; preprocess_anatomy needs a single-channel stack.")

# Raw metadata first, FISH_POLARITY only if the metadata has none.
fish_polarity_resolution = resolve_polarity(fish_folder, reviewed_polarity=FISH_POLARITY)

# Same location and name as preprocessing.canonical_preprocessing's anatomy output.
canonical_anatomy_path = fish_folder / "02_reg/00_preprocessing/2p_anatomy" / f"{fish_id}_anatomy_2P_GCaMP.nrrd"
canonical_anatomy_metadata_path = canonical_anatomy_path.with_name(f"{canonical_anatomy_path.stem}_metadata.json")

if canonical_anatomy_path.exists():
    # Never overwrite files on the data drive.
    print(f"Keeping existing {canonical_anatomy_path} (delete it by hand to rebuild).")
else:
    canonical_anatomy_record = preprocess_anatomy(
        anatomy_path=anatomy_path,
        output_path=canonical_anatomy_path,
        polarity=fish_polarity_resolution.polarity,
        source_spacing_xyz_um=(anatomy_pixel_size_um, anatomy_pixel_size_um, anatomy_z_step_um),
    )
    # Sidecar: preprocess_anatomy's own record (shapes, spacing, transforms) plus where the polarity came from.
    canonical_anatomy_metadata = {
        "fish_id": fish_id,
        "polarity": fish_polarity_resolution.polarity,
        "polarity_source": fish_polarity_resolution.source,
        **canonical_anatomy_record,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    canonical_anatomy_metadata_path.write_text(json.dumps(canonical_anatomy_metadata, indent=2))
    print(f"Wrote {canonical_anatomy_path}, shape (z, y, x) = {canonical_anatomy_record['output_shape_zyx']}")
